## Mapeo KDD → Lakehouse

En este curso, mapeamos KDD a la arquitectura Lakehouse:

| Etapa KDD | Capa Lakehouse | Descripción |
|-----------|----------------|-------------|
| Selection | **Bronze** | Datos crudos seleccionados |
| Preprocessing | **Silver** | Limpieza y validación |
| Transformation | **Gold** | Features y agregaciones |
| Data Mining | **Models** | Entrenamiento de modelos |
| Evaluation | **Reports** | Métricas y evaluación |


## Arquitectura Lakehouse

### ¿Por qué Lakehouse?

Combina lo mejor de:
- **Data Lakes**: Flexibilidad, bajo costo, formatos abiertos
- **Data Warehouses**: Estructura, calidad, performance

### Principios Clave

1. **Almacenamiento abierto**: Parquet (columnar, comprimido)
2. **Versioning**: Snapshots con particiones `run_date=YYYY-MM-DD`
3. **Capas semánticas**: Bronze → Silver → Gold
4. **Calidad de datos**: Validaciones en cada capa


## Capa Bronze: Datos Crudos

**Propósito**: Almacenar datos tal como se generan, sin transformaciones.

**Tablas**:
- `users`: Información de usuarios
- `subscriptions`: Suscripciones
- `payments`: Transacciones de pago
- `usage_logs`: Logs de uso
- `events`: Eventos de negocio (promos, outages)

**Formato**: Parquet particionado por `run_date`


## Capa Silver: Datos Validados

**Propósito**: Datos limpios, validados, listos para análisis.

**Transformaciones**:
- Eliminación de duplicados
- Validación de rangos (edad 14-90, amount > 0)
- Integridad referencial (todas las foreign keys existen)
- Normalización de timestamps a UTC
- Tipado correcto

**Validaciones** (Data Quality Gates):
- Not null checks
- Range checks
- Unique constraints
- Referential integrity


## Capa Gold: Datos Analíticos

**Propósito**: Datos optimizados para análisis y ML.

**Tablas analíticas**:
- `fact_usage_daily`: Uso agregado por día
- `fact_revenue_daily`: Ingresos por día
- `churn_features`: Features para predicción de churn
- `churn_labels`: Etiquetas de churn
- `kpis_daily`: KPIs de negocio
- `customer_segments`: Segmentos de clientes

**Features ML**:
- Ventanas temporales (7d, 30d)
- Agregaciones (sum, avg, count)
- Ratios y proporciones
- Indicadores de comportamiento


## Time-Travel con Snapshots

### Particiones run_date

Cada tabla se almacena con particiones por fecha:

```
data/bronze/users/
├── run_date=2026-01-01/
├── run_date=2026-02-01/
└── run_date=2026-03-01/
```

### Ventajas

1. **Reproducibilidad**: Mismos datos para misma fecha
2. **Auditoría**: Historial completo de cambios
3. **Rollback**: Volver a versiones anteriores
4. **Comparación**: Analizar evolución temporal

### Uso

```python
# Leer snapshot específico
df = read_parquet_snapshot(spark, path, 'users', '2026-02-01')

# Leer último snapshot
df = read_latest_snapshot(spark, path, 'users')
```


## Ejercicio: Reflexión

**Preguntas para discutir**:

1. ¿Por qué separar Bronze/Silver/Gold en lugar de una sola tabla?
2. ¿Qué beneficios trae la reproducibilidad vía snapshots?
3. ¿Cómo ayudan las validaciones de calidad en Silver?
4. ¿Qué diferencia hay entre KDD y un pipeline ETL tradicional?

**Para la próxima sesión**:
- Generar datos Bronze para tu equipo
- Explorar las tablas con PySpark
- Verificar reproducibilidad (misma seed = mismos datos)
